## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
import json
from datetime import datetime
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# ML libraries
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score, f1_score, classification_report, 
    confusion_matrix, precision_recall_fscore_support
)

# Deep learning
import torch
import torch.nn as nn
from transformers import Wav2Vec2FeatureExtractor, HubertModel

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.manifold import TSNE
import umap

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 8)

print("✓ All imports successful")
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Configuration

In [ ]:
# Paths
DATA_DIR_ADULT = Path('data/raw')  # Adult data organized by region
DATA_DIR_CHILD = Path('data/arith(child)/arith')  # Child data
FEATURES_DIR = Path('features')
RESULTS_DIR = Path('results/child_adult_study')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Audio settings
SAMPLE_RATE = 16000
SILENCE_THRESHOLD_DB = 20

# MFCC settings
N_MFCC = 40
N_FFT = 512
HOP_LENGTH = 160  # 10ms
WIN_LENGTH = 400  # 25ms

# HuBERT settings
HUBERT_MODEL = "facebook/hubert-base-ls960"
HUBERT_LAYERS = [4, 6, 8, 10]  # Key layers to analyze

# Experiment settings
RANDOM_SEED = 42
TEST_SIZE = 0.15
VAL_SIZE = 0.15

np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

print(f"Adult data: {DATA_DIR_ADULT}")
print(f"Child data: {DATA_DIR_CHILD}")
print(f"Results will be saved to: {RESULTS_DIR}")

## 3. Data Loading and Preparation

Loading audio files from adult and child directories with age group annotations.

In [ ]:
def load_metadata_with_ages():
    """
    Load audio files from separate adult and child directories.
    Adult data: data/raw/{region}/*.wav
    Child data: data/arith(child)/arith/*.wav
    """
    
    audio_files = []
    
    # Load ADULT data from regional folders
    print("Loading adult data from regional folders...")
    if DATA_DIR_ADULT.exists():
        for region_dir in DATA_DIR_ADULT.iterdir():
            if region_dir.is_dir():
                wav_files = list(region_dir.glob('*.wav'))
                print(f"  {region_dir.name}: {len(wav_files)} files")
                
                for wav_file in wav_files:
                    # Extract speaker ID from filename
                    parts = wav_file.stem.split('(')
                    if len(parts) > 1:
                        speaker_id = parts[1].rstrip(')')
                    else:
                        speaker_id = wav_file.stem
                    
                    audio_files.append({
                        'file_path': str(wav_file),
                        'speaker_id': f"adult_{speaker_id}",
                        'l1': region_dir.name,
                        'age_group': 'adult',
                        'filename': wav_file.name
                    })
    
    # Load CHILD data
    print("\nLoading child data...")
    if DATA_DIR_CHILD.exists():
        child_files = list(DATA_DIR_CHILD.glob('*.wav'))
        print(f"  arith (child): {len(child_files)} files")
        
        for wav_file in child_files:
            # Extract speaker ID from filename (e.g., arith1.wav -> 1)
            speaker_id = wav_file.stem.replace('arith', '')
            
            # NOTE: Child files don't have L1 labels in filename
            # We'll use 'child_mixed' as placeholder or you can add metadata
            audio_files.append({
                'file_path': str(wav_file),
                'speaker_id': f"child_{speaker_id}",
                'l1': 'mixed',  # Update this if you have L1 labels for children
                'age_group': 'child',
                'filename': wav_file.name
            })
    
    df = pd.DataFrame(audio_files)
    
    print("\n" + "="*60)
    print("DATA DISTRIBUTION")
    print("="*60)
    print(f"Total files: {len(df)}")
    print(f"Unique speakers: {df['speaker_id'].nunique()}")
    print(f"\nAge distribution:")
    print(df['age_group'].value_counts())
    print(f"\nL1 distribution:")
    print(df.groupby(['age_group', 'l1']).size())
    print("="*60)
    
    if len(df[df['age_group'] == 'child']) == 0:
        print("\n⚠️  WARNING: No child data found!")
        print(f"   Expected path: {DATA_DIR_CHILD}")
    elif df[df['age_group'] == 'child']['l1'].nunique() == 1:
        print("\n⚠️  NOTE: Child files don't have L1 labels.")
        print("   Cross-age evaluation will compare adult L1s only.")
        print("   Or manually add L1 labels to child metadata if available.")
    else:
        print("\n✅ Real child vs adult data loaded successfully!")
    
    return df

# Load data
metadata_df = load_metadata_with_ages()

## 4. Preprocessing: Audio Loading with Age-Specific Handling

In [ ]:
def preprocess_audio(file_path, normalize=True, trim_silence=True):
    """
    Load and preprocess audio with consistent settings for both ages.
    
    Key for cross-age: Keep preprocessing uniform but be careful with:
    - Trimming (children may have shorter valid segments)
    - Normalization (children's pitch/energy differs)
    """
    try:
        # Load
        audio, sr = librosa.load(file_path, sr=SAMPLE_RATE)
        
        # Trim silence (gentle settings to preserve short child utterances)
        if trim_silence and len(audio) > 0:
            audio, _ = librosa.effects.trim(
                audio, 
                top_db=SILENCE_THRESHOLD_DB,
                frame_length=2048,
                hop_length=512
            )
        
        # Normalize amplitude
        if normalize and len(audio) > 0:
            audio = librosa.util.normalize(audio)
        
        return audio, sr
    
    except Exception as e:
        print(f"Error loading {file_path}: {e}")
        return None, None

# Test preprocessing on a sample
sample_file = metadata_df.iloc[0]['file_path']
audio, sr = preprocess_audio(sample_file)
print(f"\nSample audio shape: {audio.shape if audio is not None else 'None'}")
print(f"Duration: {len(audio)/sr:.2f}s" if audio is not None else "Failed")

## 5. Feature Extraction
### 5A. MFCC Features (with deltas)

In [ ]:
def extract_mfcc_features(audio, sr=SAMPLE_RATE):
    """
    Extract MFCC + delta + delta2 features.
    Returns pooled (mean + std) for fixed-size representation.
    """
    if audio is None or len(audio) < WIN_LENGTH:
        return None
    
    try:
        # Compute MFCCs
        mfcc = librosa.feature.mfcc(
            y=audio, 
            sr=sr, 
            n_mfcc=N_MFCC,
            n_fft=N_FFT,
            hop_length=HOP_LENGTH,
            win_length=WIN_LENGTH
        )
        
        # Deltas
        delta = librosa.feature.delta(mfcc)
        delta2 = librosa.feature.delta(mfcc, order=2)
        
        # Stack all features
        features = np.vstack([mfcc, delta, delta2])  # (120, time)
        
        # Pool: mean + std across time → (240,) fixed vector
        pooled = np.concatenate([
            np.mean(features, axis=1),
            np.std(features, axis=1)
        ])
        
        return pooled.astype(np.float32)
    
    except Exception as e:
        print(f"MFCC extraction error: {e}")
        return None

def extract_mfcc_for_dataset(df, max_samples=None):
    """
    Extract MFCC features for all files in dataframe.
    """
    features = []
    labels = []
    ages = []
    speaker_ids = []
    
    subset = df.head(max_samples) if max_samples else df
    
    for idx, row in tqdm(subset.iterrows(), total=len(subset), desc="Extracting MFCC"):
        audio, sr = preprocess_audio(row['file_path'])
        if audio is None:
            continue
        
        feat = extract_mfcc_features(audio, sr)
        if feat is not None:
            features.append(feat)
            labels.append(row['l1'])
            ages.append(row['age_group'])
            speaker_ids.append(row['speaker_id'])
    
    return {
        'features': np.array(features),
        'labels': np.array(labels),
        'ages': np.array(ages),
        'speaker_ids': np.array(speaker_ids)
    }

# Extract MFCC features (limit for demo; remove max_samples for full run)
print("\nExtracting MFCC features...")
mfcc_data = extract_mfcc_for_dataset(metadata_df, max_samples=500)
print(f"\nMFCC features shape: {mfcc_data['features'].shape}")
print(f"Adult samples: {(mfcc_data['ages'] == 'adult').sum()}")
print(f"Child samples: {(mfcc_data['ages'] == 'child').sum()}")

### 5B. HuBERT Embeddings (Layer-wise)

In [ ]:
def load_hubert_model(device='cuda' if torch.cuda.is_available() else 'cpu'):
    """
    Load HuBERT model and feature extractor for feature extraction.
    """
    print(f"Loading HuBERT model on {device}...")
    feature_extractor = Wav2Vec2FeatureExtractor.from_pretrained(HUBERT_MODEL)
    model = HubertModel.from_pretrained(
        HUBERT_MODEL, 
        output_hidden_states=True
    ).to(device)
    model.eval()
    return feature_extractor, model, device

def extract_hubert_features(audio, feature_extractor, model, device, layer=4):
    """
    Extract HuBERT embeddings from specified layer with mean pooling.
    """
    if audio is None or len(audio) < 400:
        return None
    
    try:
        # Pad short audio
        if len(audio) < 400:
            audio = np.pad(audio, (0, 400 - len(audio)))
        
        # Process
        inputs = feature_extractor(
            audio, 
            sampling_rate=SAMPLE_RATE, 
            return_tensors="pt", 
            padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        
        # Extract
        with torch.no_grad():
            outputs = model(**inputs)
            hidden_states = outputs.hidden_states
            
            # Get specified layer
            layer_output = hidden_states[layer]  # (batch, time, dim)
            pooled = torch.mean(layer_output, dim=1).cpu().numpy()[0]
        
        return pooled.astype(np.float32)
    
    except Exception as e:
        print(f"HuBERT extraction error: {e}")
        return None

def extract_hubert_for_dataset(df, layer=4, max_samples=None):
    """
    Extract HuBERT features for all files.
    """
    feature_extractor, model, device = load_hubert_model()
    
    features = []
    labels = []
    ages = []
    speaker_ids = []
    
    subset = df.head(max_samples) if max_samples else df
    
    for idx, row in tqdm(subset.iterrows(), total=len(subset), 
                         desc=f"Extracting HuBERT Layer {layer}"):
        audio, sr = preprocess_audio(row['file_path'])
        if audio is None:
            continue
        
        feat = extract_hubert_features(audio, feature_extractor, model, device, layer)
        if feat is not None:
            features.append(feat)
            labels.append(row['l1'])
            ages.append(row['age_group'])
            speaker_ids.append(row['speaker_id'])
    
    return {
        'features': np.array(features),
        'labels': np.array(labels),
        'ages': np.array(ages),
        'speaker_ids': np.array(speaker_ids)
    }

# Extract HuBERT features for key layer (limit for demo)
print("\nExtracting HuBERT features...")
hubert_data = extract_hubert_for_dataset(metadata_df, layer=4, max_samples=500)
print(f"\nHuBERT features shape: {hubert_data['features'].shape}")
print(f"Adult samples: {(hubert_data['ages'] == 'adult').sum()}")
print(f"Child samples: {(hubert_data['ages'] == 'child').sum()}")

In [ ]:
# Since first 500 are all adults, let's extract features for ALL child files + sample of adults
print("\n" + "="*60)
print("EXTRACTING FEATURES FOR CHILD VS ADULT ANALYSIS")
print("="*60)

# Separate adult and child data
adult_df = metadata_df[metadata_df['age_group'] == 'adult']
child_df = metadata_df[metadata_df['age_group'] == 'child']

print(f"\nAdult samples available: {len(adult_df)}")
print(f"Child samples available: {len(child_df)}")

# Sample adults (take 300 for faster processing, or remove sample() for all)
adult_sample = adult_df.sample(n=min(300, len(adult_df)), random_state=42)
combined_df = pd.concat([adult_sample, child_df]).reset_index(drop=True)

print(f"\nProcessing {len(adult_sample)} adults + {len(child_df)} children = {len(combined_df)} total")

# Extract MFCC for combined dataset
print("\n--- Extracting MFCC ---")
mfcc_data = extract_mfcc_for_dataset(combined_df)
print(f"MFCC shape: {mfcc_data['features'].shape}")
print(f"Adult: {(mfcc_data['ages'] == 'adult').sum()}, Child: {(mfcc_data['ages'] == 'child').sum()}")

# Extract HuBERT for combined dataset
print("\n--- Extracting HuBERT ---")
hubert_data = extract_hubert_for_dataset(combined_df, layer=4)
print(f"HuBERT shape: {hubert_data['features'].shape}")
print(f"Adult: {(hubert_data['ages'] == 'adult').sum()}, Child: {(hubert_data['ages'] == 'child').sum()}")

## 6. Data Splitting Strategy
### Speaker-wise splits to prevent data leakage

In [ ]:
def create_cross_age_splits(data):
    """
    Create train/val/test splits for cross-age evaluation.
    
    Returns:
        - adult_train, adult_val, adult_test (indices)
        - child_train, child_val, child_test (indices)
    """
    ages = data['ages']
    speaker_ids = data['speaker_ids']
    labels = data['labels']
    
    # Separate by age
    adult_mask = ages == 'adult'
    child_mask = ages == 'child'
    
    # Get unique speakers for each age group
    adult_speakers = np.unique(speaker_ids[adult_mask])
    child_speakers = np.unique(speaker_ids[child_mask])
    
    # Split adult speakers: 70% train, 15% val, 15% test
    np.random.shuffle(adult_speakers)
    n_adult = len(adult_speakers)
    adult_train_spk = adult_speakers[:int(0.7*n_adult)]
    adult_val_spk = adult_speakers[int(0.7*n_adult):int(0.85*n_adult)]
    adult_test_spk = adult_speakers[int(0.85*n_adult):]
    
    # Split child speakers similarly
    np.random.shuffle(child_speakers)
    n_child = len(child_speakers)
    child_train_spk = child_speakers[:int(0.7*n_child)]
    child_val_spk = child_speakers[int(0.7*n_child):int(0.85*n_child)]
    child_test_spk = child_speakers[int(0.85*n_child):]
    
    # Get indices
    def get_indices(speakers, speaker_ids):
        return np.where(np.isin(speaker_ids, speakers))[0]
    
    splits = {
        'adult_train': get_indices(adult_train_spk, speaker_ids),
        'adult_val': get_indices(adult_val_spk, speaker_ids),
        'adult_test': get_indices(adult_test_spk, speaker_ids),
        'child_train': get_indices(child_train_spk, speaker_ids),
        'child_val': get_indices(child_val_spk, speaker_ids),
        'child_test': get_indices(child_test_spk, speaker_ids),
    }
    
    # Print split statistics
    print("\n" + "="*60)
    print("DATA SPLITS")
    print("="*60)
    for name, indices in splits.items():
        print(f"{name:20s}: {len(indices):4d} samples")
    print("="*60)
    
    return splits

# Create splits
mfcc_splits = create_cross_age_splits(mfcc_data)
hubert_splits = create_cross_age_splits(hubert_data)

## 7. Experiment 1: Train on Adults → Test on Children (MFCC)

In [ ]:
def train_and_evaluate_cross_age(data, splits, feature_name='MFCC'):
    """
    Train on adult data, evaluate on both adults (within-age) and children (cross-age).
    """
    results = {}
    
    # Prepare data
    X = data['features']
    y = data['labels']
    
    # Encode labels
    le = LabelEncoder()
    y_encoded = le.fit_transform(y)
    
    # Get splits
    X_adult_train = X[splits['adult_train']]
    y_adult_train = y_encoded[splits['adult_train']]
    X_adult_test = X[splits['adult_test']]
    y_adult_test = y_encoded[splits['adult_test']]
    X_child_test = X[splits['child_test']]
    y_child_test = y_encoded[splits['child_test']]
    
    # Standardize
    scaler = StandardScaler()
    X_adult_train = scaler.fit_transform(X_adult_train)
    X_adult_test = scaler.transform(X_adult_test)
    X_child_test = scaler.transform(X_child_test)
    
    print(f"\n{'='*60}")
    print(f"EXPERIMENT: Train on Adults → Test on Adults & Children ({feature_name})")
    print(f"{'='*60}")
    
    # Train Random Forest
    print("\nTraining Random Forest...")
    rf = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=RANDOM_SEED, n_jobs=-1)
    rf.fit(X_adult_train, y_adult_train)
    
    # Evaluate on adults (within-age baseline)
    y_adult_pred = rf.predict(X_adult_test)
    adult_acc = accuracy_score(y_adult_test, y_adult_pred)
    adult_f1 = f1_score(y_adult_test, y_adult_pred, average='macro')
    
    print(f"\n📊 WITHIN-AGE (Adult → Adult):")
    print(f"   Accuracy: {adult_acc:.4f}")
    print(f"   Macro F1: {adult_f1:.4f}")
    
    # Evaluate on children (cross-age)
    y_child_pred = rf.predict(X_child_test)
    
    # Check if we have child samples
    if len(y_child_test) == 0:
        print(f"\n⚠️  No child test samples available!")
        child_acc = 0.0
        child_f1 = 0.0
    else:
        child_acc = accuracy_score(y_child_test, y_child_pred)
        # Use zero_division parameter for f1_score
        child_f1 = f1_score(y_child_test, y_child_pred, average='macro', zero_division=0)
        
        print(f"\n📊 CROSS-AGE (Adult → Child):")
        print(f"   Accuracy: {child_acc:.4f}")
        print(f"   Macro F1: {child_f1:.4f}")
        print(f"   Drop: {(adult_acc - child_acc)*100:.2f}% absolute")
        print(f"\n   Note: Children have '{le.inverse_transform([y_child_test[0]])[0]}' L1 label only")
        print(f"   Predicted classes: {[le.inverse_transform([p])[0] for p in y_child_pred]}")
    
    # Store results
    results[feature_name] = {
        'within_age_accuracy': float(adult_acc),
        'within_age_f1': float(adult_f1),
        'cross_age_accuracy': float(child_acc),
        'cross_age_f1': float(child_f1),
        'accuracy_drop': float(adult_acc - child_acc),
        'child_samples': len(y_child_test)
    }
    
    return results, rf, scaler, le, y_adult_test, y_adult_pred, y_child_test, y_child_pred

# Run experiment with MFCC
mfcc_results, mfcc_model, mfcc_scaler, mfcc_le, *mfcc_preds = train_and_evaluate_cross_age(
    mfcc_data, mfcc_splits, 'MFCC'
)

## 8. Experiment 2: Train on Adults → Test on Children (HuBERT)

In [ ]:
# Run experiment with HuBERT
hubert_results, hubert_model, hubert_scaler, hubert_le, *hubert_preds = train_and_evaluate_cross_age(
    hubert_data, hubert_splits, 'HuBERT'
)

## 9. Comparison: MFCC vs HuBERT Cross-Age Robustness

In [ ]:
# Combine results
all_results = {**mfcc_results, **hubert_results}

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot 1: Accuracy comparison
features = list(all_results.keys())
within_age = [all_results[f]['within_age_accuracy'] for f in features]
cross_age = [all_results[f]['cross_age_accuracy'] for f in features]

x = np.arange(len(features))
width = 0.35

axes[0].bar(x - width/2, within_age, width, label='Within-age (Adult→Adult)', color='steelblue')
axes[0].bar(x + width/2, cross_age, width, label='Cross-age (Adult→Child)', color='coral')
axes[0].set_ylabel('Accuracy')
axes[0].set_title('Cross-Age Generalization: MFCC vs HuBERT')
axes[0].set_xticks(x)
axes[0].set_xticklabels(features)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)

# Plot 2: Accuracy drop
drops = [all_results[f]['accuracy_drop'] * 100 for f in features]
colors = ['coral' if d > 0 else 'green' for d in drops]
axes[1].bar(features, drops, color=colors, alpha=0.7)
axes[1].set_ylabel('Accuracy Drop (%)')
axes[1].set_title('Robustness: Lower Drop = Better Cross-Age Transfer')
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=0.8)
axes[1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'mfcc_vs_hubert_cross_age.png', dpi=300, bbox_inches='tight')
plt.show()

print("\n" + "="*60)
print("SUMMARY: MFCC vs HuBERT Cross-Age Robustness")
print("="*60)
for feat in features:
    print(f"\n{feat}:")
    print(f"  Within-age accuracy: {all_results[feat]['within_age_accuracy']:.4f}")
    print(f"  Cross-age accuracy:  {all_results[feat]['cross_age_accuracy']:.4f}")
    print(f"  Accuracy drop:       {all_results[feat]['accuracy_drop']*100:.2f}%")
print("="*60)

## 10. Visualization: Confusion Matrices

In [ ]:
def plot_confusion_matrices(y_true_adult, y_pred_adult, y_true_child, y_pred_child, 
                           labels, title_prefix=''):
    """
    Plot side-by-side confusion matrices for within-age and cross-age.
    """
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    
    # Within-age (Adult → Adult)
    cm_adult = confusion_matrix(y_true_adult, y_pred_adult)
    sns.heatmap(cm_adult, annot=True, fmt='d', cmap='Blues', 
                xticklabels=labels, yticklabels=labels, ax=axes[0])
    axes[0].set_title(f'{title_prefix} Within-Age (Adult→Adult)')
    axes[0].set_ylabel('True Label')
    axes[0].set_xlabel('Predicted Label')
    
    # Cross-age (Adult → Child)
    cm_child = confusion_matrix(y_true_child, y_pred_child)
    sns.heatmap(cm_child, annot=True, fmt='d', cmap='Reds', 
                xticklabels=labels, yticklabels=labels, ax=axes[1])
    axes[1].set_title(f'{title_prefix} Cross-Age (Adult→Child)')
    axes[1].set_ylabel('True Label')
    axes[1].set_xlabel('Predicted Label')
    
    plt.tight_layout()
    return fig

# Plot for MFCC
fig_mfcc = plot_confusion_matrices(
    mfcc_preds[0], mfcc_preds[1], mfcc_preds[2], mfcc_preds[3],
    mfcc_le.classes_, 'MFCC:'
)
fig_mfcc.savefig(RESULTS_DIR / 'mfcc_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

# Plot for HuBERT
fig_hubert = plot_confusion_matrices(
    hubert_preds[0], hubert_preds[1], hubert_preds[2], hubert_preds[3],
    hubert_le.classes_, 'HuBERT:'
)
fig_hubert.savefig(RESULTS_DIR / 'hubert_confusion_matrices.png', dpi=300, bbox_inches='tight')
plt.show()

## 11. Embedding Visualization: t-SNE (Age Separation)

In [ ]:
def visualize_embeddings_tsne(data, splits, feature_name='Features'):
    """
    t-SNE visualization colored by L1 and shaped by age group.
    """
    # Sample for visualization (t-SNE is slow on large datasets)
    sample_size = min(500, len(data['features']))
    indices = np.random.choice(len(data['features']), sample_size, replace=False)
    
    X_sample = data['features'][indices]
    y_sample = data['labels'][indices]
    age_sample = data['ages'][indices]
    
    # Standardize
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sample)
    
    # t-SNE
    print(f"\nComputing t-SNE for {feature_name}...")
    tsne = TSNE(n_components=2, random_state=RANDOM_SEED, perplexity=30)
    X_tsne = tsne.fit_transform(X_scaled)
    
    # Plot
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Color by L1
    unique_l1s = np.unique(y_sample)
    colors = plt.cm.tab10(np.linspace(0, 1, len(unique_l1s)))
    
    for i, l1 in enumerate(unique_l1s):
        mask = y_sample == l1
        axes[0].scatter(
            X_tsne[mask, 0], X_tsne[mask, 1],
            c=[colors[i]], label=l1, alpha=0.6, s=30
        )
    axes[0].set_title(f't-SNE: {feature_name} colored by L1')
    axes[0].legend()
    axes[0].grid(alpha=0.3)
    
    # Shape by age
    adult_mask = age_sample == 'adult'
    child_mask = age_sample == 'child'
    
    axes[1].scatter(
        X_tsne[adult_mask, 0], X_tsne[adult_mask, 1],
        c='steelblue', marker='o', label='Adult', alpha=0.6, s=30
    )
    axes[1].scatter(
        X_tsne[child_mask, 0], X_tsne[child_mask, 1],
        c='coral', marker='^', label='Child', alpha=0.6, s=30
    )
    axes[1].set_title(f't-SNE: {feature_name} colored by Age Group')
    axes[1].legend()
    axes[1].grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / f'{feature_name.lower()}_tsne.png', dpi=300, bbox_inches='tight')
    plt.show()

# Visualize MFCC
visualize_embeddings_tsne(mfcc_data, mfcc_splits, 'MFCC')

# Visualize HuBERT
visualize_embeddings_tsne(hubert_data, hubert_splits, 'HuBERT')

## 12. Layer-wise HuBERT Analysis (Optional Deep Dive)

In [ ]:
def analyze_hubert_layers(df, layers=[4, 6, 8, 10], max_samples=300):
    """
    Extract features from multiple HuBERT layers and compare cross-age robustness.
    """
    layer_results = {}
    
    for layer in layers:
        print(f"\n{'='*60}")
        print(f"Analyzing HuBERT Layer {layer}")
        print(f"{'='*60}")
        
        # Extract features for this layer
        data = extract_hubert_for_dataset(df, layer=layer, max_samples=max_samples)
        splits = create_cross_age_splits(data)
        
        # Train and evaluate
        results, *_ = train_and_evaluate_cross_age(
            data, splits, f'HuBERT_Layer{layer}'
        )
        
        layer_results[f'Layer_{layer}'] = results[f'HuBERT_Layer{layer}']
    
    # Plot comparison
    fig, ax = plt.subplots(figsize=(10, 6))
    
    layers_list = list(layer_results.keys())
    within_age = [layer_results[l]['within_age_accuracy'] for l in layers_list]
    cross_age = [layer_results[l]['cross_age_accuracy'] for l in layers_list]
    
    x = np.arange(len(layers_list))
    width = 0.35
    
    ax.plot(x, within_age, 'o-', label='Within-age', linewidth=2, markersize=8)
    ax.plot(x, cross_age, 's-', label='Cross-age', linewidth=2, markersize=8)
    ax.set_xlabel('HuBERT Layer')
    ax.set_ylabel('Accuracy')
    ax.set_title('Layer-wise Cross-Age Robustness')
    ax.set_xticks(x)
    ax.set_xticklabels(layers_list)
    ax.legend()
    ax.grid(alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'hubert_layerwise_cross_age.png', dpi=300, bbox_inches='tight')
    plt.show()
    
    return layer_results

# Run layer-wise analysis (commented out by default due to compute cost)
# Uncomment to run full layer analysis:
# layer_results = analyze_hubert_layers(metadata_df, layers=HUBERT_LAYERS, max_samples=300)

## 13. Save Complete Results

In [ ]:
# Compile final results
final_results = {
    'timestamp': datetime.now().isoformat(),
    'configuration': {
        'sample_rate': SAMPLE_RATE,
        'n_mfcc': N_MFCC,
        'hubert_model': HUBERT_MODEL,
        'random_seed': RANDOM_SEED
    },
    'data_statistics': {
        'total_samples': int(len(metadata_df)),
        'adult_samples': int((metadata_df['age_group'] == 'adult').sum()),
        'child_samples': int((metadata_df['age_group'] == 'child').sum()),
        'unique_speakers': int(metadata_df['speaker_id'].nunique()),
        'l1_distribution': {k: int(v) for k, v in metadata_df['l1'].value_counts().to_dict().items()}
    },
    'results': all_results,
    'key_findings': {
        'best_feature': max(all_results, key=lambda x: all_results[x]['cross_age_accuracy']),
        'most_robust': min(all_results, key=lambda x: all_results[x]['accuracy_drop']),
        'mfcc_drop_percent': all_results['MFCC']['accuracy_drop'] * 100,
        'hubert_drop_percent': all_results['HuBERT']['accuracy_drop'] * 100
    }
}

# Save to JSON
with open(RESULTS_DIR / 'child_adult_generalization_results.json', 'w') as f:
    json.dump(final_results, f, indent=2)

print("\n" + "="*60)
print("✅ RESULTS SAVED")
print("="*60)
print(f"Location: {RESULTS_DIR}")
print("\nFiles:")
for file in RESULTS_DIR.glob('*'):
    print(f"  - {file.name}")
print("="*60)

## 14. Summary Report

In [ ]:
print("\n" + "="*70)
print("CHILD VS ADULT GENERALIZATION STUDY - FINAL REPORT")
print("="*70)

print("\n📊 DATA OVERVIEW:")
print(f"   Total samples: {final_results['data_statistics']['total_samples']}")
print(f"   Adult samples: {final_results['data_statistics']['adult_samples']}")
print(f"   Child samples: {final_results['data_statistics']['child_samples']}")
print(f"   Unique speakers: {final_results['data_statistics']['unique_speakers']}")

print("\n🎯 KEY FINDINGS:")
print(f"   Best cross-age feature: {final_results['key_findings']['best_feature']}")
print(f"   Most robust (lowest drop): {final_results['key_findings']['most_robust']}")

print("\n📈 PERFORMANCE COMPARISON:")
print("\n   MFCC:")
print(f"      Within-age: {all_results['MFCC']['within_age_accuracy']:.4f}")
print(f"      Cross-age:  {all_results['MFCC']['cross_age_accuracy']:.4f}")
print(f"      Drop:       {final_results['key_findings']['mfcc_drop_percent']:.2f}%")

print("\n   HuBERT:")
print(f"      Within-age: {all_results['HuBERT']['within_age_accuracy']:.4f}")
print(f"      Cross-age:  {all_results['HuBERT']['cross_age_accuracy']:.4f}")
print(f"      Drop:       {final_results['key_findings']['hubert_drop_percent']:.2f}%")

# Determine winner
mfcc_drop = all_results['MFCC']['accuracy_drop']
hubert_drop = all_results['HuBERT']['accuracy_drop']

print("\n💡 CONCLUSION:")
if hubert_drop < mfcc_drop:
    improvement = (mfcc_drop - hubert_drop) * 100
    print(f"   HuBERT shows {improvement:.2f}% better cross-age robustness than MFCC.")
    print("   ✅ HuBERT embeddings are more age-invariant and generalize better.")
else:
    print("   Both features show similar cross-age performance.")
    print("   Consider ensemble or domain adaptation techniques.")

print("\n📝 RECOMMENDATIONS:")
print("   1. Use HuBERT embeddings for better cross-age robustness")
print("   2. Fine-tune on small child dataset to further reduce drop")
print("   3. Try domain-adversarial training to learn age-invariant features")
print("   4. Analyze per-class performance to identify challenging L1s")
print("   5. Consider data augmentation (pitch/speed) for age variation")

print("\n" + "="*70)
print("⚠️  NOTE: Results use simulated age groups for pipeline demonstration.")
print("   For production analysis, collect real child vs adult metadata.")
print("="*70 + "\n")

## 15. Conclusions

### Key Findings:
1. **HuBERT shows better cross-age robustness** - 6.16% smaller accuracy drop compared to MFCC
2. **Both features struggle with cross-age generalization** - Models trained on adults fail on children
3. **Age-specific acoustic differences** are clearly visible in t-SNE embeddings
4. **Within-age performance is strong** - Both MFCC (95.7%) and HuBERT (89.6%) achieve high accuracy on adult test data

### Future Work:
- Collect more child data with L1 labels for detailed per-class analysis
- Implement domain adaptation techniques (e.g., domain-adversarial training)
- Apply data augmentation (pitch/speed variation) to improve age robustness
- Fine-tune models on small child dataset using transfer learning
- Explore multi-task learning jointly predicting L1 and age group

---

## Note: Why the Code is in Scripts

I ran into issues running everything in this notebook - kept crashing when processing all the audio files. So I moved the working code to Python scripts in the `scripts/` folder.

The cells below show how to use those scripts if you want to run the code yourself.

### Main Scripts

1. `scripts/persistent_hubert.py` - Loads HuBERT model once instead of reloading every time
2. `scripts/predict_backend.py` - Main prediction logic with unknown detection  
3. `scripts/train_final_model.py` - Trains the Random Forest model
4. `app.py` - Web interface (Gradio)

### Example: Using the Prediction Backend

The backend module handles everything - cached features, live extraction, calibration, etc.

In [ ]:
# Example: Using predict_backend for inference
import sys
sys.path.insert(0, 'scripts')

from predict_backend import predict_from_path

# Test with a sample audio file
audio_path = "data/raw/karnataka/Karnataka_speaker_02_1 (744).wav"
result = predict_from_path(audio_path)

print(f"Predicted Label: {result['predicted_label']}")
print(f"Confidence: {result['confidence']:.4f}")
print(f"Unknown Flag: {result['unknown']}")
print(f"Mahalanobis Distance: {result.get('mahal_distance', 'N/A')}")
print(f"\nTop 3 Predictions:")
sorted_probs = sorted(result['all_probabilities'].items(), key=lambda x: x[1], reverse=True)[:3]
for label, prob in sorted_probs:
    print(f"  {label}: {prob:.4f}")

### Running from Terminal

All the scripts work from command line:

```bash
# Extract features
python scripts/batch_extract_features.py

# Train model  
python scripts/train_final_model.py

# Make prediction
python scripts/predict.py "audio.wav"

# Launch web app
python app.py
```

### How the Persistent Model Works

This was the key fix - loading HuBERT once instead of every time:

In [ ]:
# Key idea: Load model ONCE and reuse it
# (from scripts/persistent_hubert.py)

_hubert_model = None

def init_hubert():
    """Load HuBERT model once, reuse forever"""
    global _hubert_model
    if _hubert_model is not None:
        return  # Already loaded
    
    from transformers import HubertModel
    _hubert_model = HubertModel.from_pretrained("facebook/hubert-base-ls960")
    _hubert_model.eval()

def extract_live_features(audio_path):
    """Extract features using the persistent model"""
    init_hubert()
    # ... audio loading ...
    with torch.no_grad():
        outputs = _hubert_model(...)
        return pooled_features

# This fixed the feature mismatch problem!

### Detecting Unknown Samples

Added Mahalanobis distance to flag samples that don't fit any class:

In [ ]:
# Compute stats for each class during training
# (from scripts/train_final_model.py)

class_means = {}
class_covs = {}

for region in ['andhra_pradesh', 'gujarat', ...]:
    samples = X_train[y_train == region]
    class_means[region] = samples.mean(axis=0)
    class_covs[region] = np.cov(samples, rowvar=False)

# During prediction, check if sample is too far
from scipy.spatial.distance import mahalanobis

dist = mahalanobis(sample, class_means[predicted], inv(class_covs[predicted]))
if dist > threshold:
    print("Warning: This sample might not fit any known class")

### Summary

**Problems I had:**
- Notebook kept crashing with 3700+ audio files
- HuBERT features were slightly different each time I loaded the model
- Couldn't reproduce results reliably

**Solutions:**
- Moved code to Python scripts
- Load HuBERT model once and reuse it
- Added calibration to align live and cached features
- Added unknown detection for out-of-distribution samples

**Results:**
- Accuracy: 94.83% → 99.73%
- Speed: 10-15s → 2-3s per file
- Memory: Crashes → Stable
- Reproducibility: Variable → Fixed

The notebooks show my exploration work. The scripts are the production code.